## Domain Adaptation

Usually, you can fine-tune a pretrained model directly on your task data.

If your data uses specialized language, such as legal or scientific text, first continue training the model on that type of text. This helps it learn the domain’s vocabulary and writing style.

This step is called **domain adaptation**. Afterward, fine-tune the model for your specific task.

In [84]:
CHECKPOINT = "distilbert/distilbert-base-uncased"

Although BERT and RoBERTa models are widely used, this example uses **DistilBERT**, a smaller model that trains faster while often retaining similar performance on downstream tasks.

DistilBERT was created using **knowledge distillation**: a larger “teacher” model, such as BERT, guides the training of a smaller “student” model. The details of this technique are beyond this section.

In [85]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForMaskedLM.from_pretrained(CHECKPOINT)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 896.34it/s]


In [86]:
model.num_parameters()

66985530

In [87]:
text = 'This is a great [MASK]'

In [88]:
import torch

In [89]:
inputs = tokenizer(text, return_tensors='pt')
token_logits = model(**inputs).logits

In [90]:
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

In [91]:
for token in top_5_tokens:
    print(f"> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}")

> This is a great !
> This is a great .
> This is a great deal
> This is a great adventure
> This is a great ;


In [92]:
from datasets import load_dataset

In [93]:
imdb_dataset = load_dataset("stanfordnlp/imdb")
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [94]:
sample = imdb_dataset['train'].shuffle(seed=42).select(range(3))

In [95]:
for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...'
'>>> Label: 1'

'>>> Review: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stu

In [96]:
for split in ["train", "test", "unsupervised"]:
    print(split, set(imdb_dataset[split]["label"]))

train {0, 1}
test {0, 1}
unsupervised {-1}


In [97]:
def tokenize_function(examples):
    results = tokenizer(
        examples["text"],
        return_offsets_mapping=True,
    )

    if tokenizer.is_fast:
        results["word_ids"] = [
            results.word_ids(i)
            for i in range(len(results["input_ids"]))
        ]

    return results


In [98]:
tokenized_dataset = imdb_dataset.map(tokenize_function, batched=True, remove_columns=['text', 'label'])

Map: 100%|██████████| 50000/50000 [00:45<00:00, 1101.81 examples/s]


In [99]:
chunk_size = 128

Note that using a small chunk size can be detrimental in real-world scenarios, so you should use a size that corresponds to the use case you will apply your model to.

In [100]:
tokenized_samples = tokenized_dataset["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Review {idx} length: {len(sample)}'")

'>>> Review 0 length: 363'
'>>> Review 1 length: 304'
'>>> Review 2 length: 133'


In [101]:
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // chunk_size) * chunk_size
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()
    return result

In [102]:
lm_datasets = tokenized_dataset.map(group_texts, batched=True)
lm_datasets

Map: 100%|██████████| 50000/50000 [04:34<00:00, 182.30 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [103]:
tokenizer.decode(lm_datasets["train"][1]["input_ids"])

"as the vietnam war and race issues in the united states. in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men. < br / > < br / > what kills me about i am curious - yellow is that 40 years ago, this was considered pornographic. really, the sex and nudity scenes are few and far between, even then it ' s not shot like some cheaply made porno. while my countrymen mind find it shocking, in reality sex and nudity are a major staple in swedish cinema. even ingmar bergman,"

In [104]:
from transformers import DataCollatorForWholeWordMask

In [105]:
data_collator = DataCollatorForWholeWordMask(tokenizer=tokenizer, mlm_probability=0.15)

In [106]:
samples = [lm_datasets["train"][i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it [MASK] first released in 1967. i also heard that at first it was seized by u. s. customs if it ever tried to [MASK] this [MASK] [MASK] therefore being a fan of [MASK] considered " controversial " i really had to see this for myself. < br [MASK] [MASK] [MASK] [MASK] / > the plot is centered around a [MASK] swedish drama student named lena who [MASK] to learn everything she [MASK] about life. in particular she [MASK] to [MASK] her attentions to [MASK] some sort of [MASK] on what the average swede thought about certaincule issues such'

'>>> as the [MASK] war and race loyal [MASK] the united states. in [MASK] asking politicians and ordinary denizens of stockholm [MASK] their [MASK] on politics, she has sex with [MASK] drama teacher, classmates, and married men. < br / > < br / [MASK] [MASK] kills me about i am curious - yellow is that 40 years ago, this was considered p

In [107]:
train_size = 10000
test_size = int(train_size * 0.1)

downsampled_dataset = lm_datasets['train'].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)

In [108]:
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

In [109]:
from huggingface_hub import notebook_login

notebook_login()

In [110]:
from transformers import TrainingArguments, Trainer

batch_size = 64
logging_steps = len(downsampled_dataset['train']) // batch_size
model_name = CHECKPOINT.split('/')[-1]

train_args = TrainingArguments(
    output_dir=f'f{model_name}-finetuned-imdb',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps
)

In [111]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=downsampled_dataset['train'],
    eval_dataset=downsampled_dataset['test'],
    data_collator=data_collator,
)

In [112]:
trainer.train()

IndexError: too many indices for array: array is 0-dimensional, but 3 were indexed